In [1]:
from huggingface_hub import login, upload_folder, hf_hub_download, HfApi
from datasets import Dataset
from google.colab import userdata
import re
import unicodedata
from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, trainers

In [2]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
file_path = hf_hub_download(
    repo_id="xkas2001/uzbek-language-dataset",
    filename="custom-uzbek/parsed_with_imlo_without_emoji.txt",
    repo_type="dataset",
    revision="main"
)

custom-uzbek/parsed_with_imlo_without_em(…):   0%|          | 0.00/546M [00:00<?, ?B/s]

In [4]:
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

In [ ]:
print(f'len of text: {len(text)}')
uniques = set(text)
print(f'len_uniques: {len(uniques)}')

len of text: 528402040
len_uniques: 440


In [ ]:
print(uniques)

{'真', 'å', '‚', 'O', 'ц', 'Ю', 'إ', 'ǧ', 'ƒ', 'j', 'ا', 'ん', '^', 'A', 'ө', 'α', '0', '‒', 'ɑ', '‛', 'ſ', 'ґ', 'ӣ', '问', 'ֲ', 'ғ', 'Ф', 'ċ', 'щ', '\u200e', 'κ', 'أ', '\u200a', 'ツ', 'د', '버', '\u2009', 'ض', 'ɔ', '郎', '\u200c', 'е', '∙', 'Ҫ', 'm', 'ī', 'ł', 'ָ', 'ж', '″', '五', 'і', '中', '–', 'Ә', '¯', '⁰', 'э', '\u200d', '̃', 'ή', 'ل', '′', 'æ', '_', 'K', 'L', 'É', 'Ҷ', '\u200b', 'q', 'Ғ', 'Ḡ', 'ˡ', '\u0605', '˘', 'î', 'û', 'ɪ', 'Й', 'y', 'ᶟ', 'ÿ', 'к', 'Ӯ', 'ᶇ', 'ž', '学', 'ֺ', 'υ', 'ˌ', '9', ']', '⁃', '\t', 'Ү', '-', '카', 'U', 'í', 'ӽ', 'д', '↓', '\u2028', 'Я', 'ć', 'ʒ', '̧', 'Ӧ', 'К', 'ß', 'u', '\\', 'ک', '1', 'ʊ', 'º', 'о', 'ى', 'Ε', 'ṣ', '̅', 'e', 'Ё', 'Ǧ', 'ك', '\u202c', 'ầ', 'ṡ', 'ʾ', 'ؘ', 'ְ', 'C', 'ô', '\x8d', '§', 'ق', 'ج', 'З', '<', 'م', 'M', 'Т', '3', 'ǵ', '고', 'Y', 'n', 'Ұ', '\n', 'Ǵ', 'Ј', 'ю', 'ъ', 'ĝ', 'ظ', 'č', 'ش', 'ф', '경', 'Ĉ', 'h', 'ɒ', 'ʃ', 'ј', 'н', 'S', 'ï', 'π', '¿', '´', 'Ы', 'Х', 'І', 'ر', 'Ă', 'چ', 'ɜ', '|', 'с', 'k', 'N', 'ь', '2', '题', 'ē', 'б', 'ў', '֥', 'J'

In [ ]:
# --- 1. Allowed characters regex ---
ALLOWED_PATTERN = re.compile(
    r"[^a-zA-Z0-9ʻ\-.,:;!?()%/\n ]"
)

# --- 2. Normalize apostrophes to one symbol (U+02BB) ---
def normalize_apostrophes(text: str) -> str:
    apostrophes = ["’", "‘", "ʼ", "'", "`", "´"]
    for ch in apostrophes:
        text = text.replace(ch, "ʻ")  # U+02BB
    return text

# --- 3. Remove invisible / zero-width Unicode ---
INVISIBLE_CHARS = [
    "\u200b", "\u200c", "\u200d",
    "\u200e", "\u200f",
    "\ufeff",
    "\u202a", "\u202c", "\u2028",
    "\u2060"
]

def remove_invisible(text: str) -> str:
    for ch in INVISIBLE_CHARS:
        text = text.replace(ch, "")
    return text

# --- 4. Replace tabs and backslashes ---
def replace_tabs_and_backslashes(text: str) -> str:
    text = text.replace("\t", " ")  # tab → space
    text = text.replace("\\", "")   # remove backslash
    return text

# --- 5. Collapse multiple spaces and newlines ---
def collapse_whitespace(text: str) -> str:
    # Collapse multiple spaces
    text = re.sub(r"[ ]{2,}", " ", text)
    # Collapse multiple newlines (max 2)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# --- 6. Main cleaning function ---
def clean_text(text: str) -> str:
    # Unicode normalization
    text = unicodedata.normalize("NFC", text)
    text = remove_invisible(text)
    text = normalize_apostrophes(text)
    text = replace_tabs_and_backslashes(text)
    text = ALLOWED_PATTERN.sub("", text)
    text = collapse_whitespace(text)
    return text

# --- 7. File-level cleaner ---
def clean_file(input_path: str, output_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()
    cleaned = clean_text(text)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned)
        unique_cl = set(cleaned)
        print(f'len of text: {len(cleaned)}')
        print(f'uniques: {unique_cl}')
        print(f'len_uniques: {len(unique_cl)}')

In [6]:
clean_file(file_path, "cleaned_text.txt")

len of text: 526921719
uniques: {'x', 'L', 'v', 'k', 'r', 'u', '\n', 'X', 'P', '9', 'a', 'c', '5', 'R', '8', 'l', 'n', 'j', 'g', 'F', 'K', 'B', 'w', 'b', 'Z', 'W', 'o', 'Q', 'm', 'z', 'A', 'e', 'y', 'h', '2', 'U', 'V', 'C', '6', 'T', 'p', 'I', 't', '7', 'i', 'O', 'M', 'f', 'Y', 'N', 'ʻ', 'D', '1', '3', 'S', 'q', 'd', '-', 'H', 'E', 's', 'G', '0', '4', ' ', 'J'}
len_uniques: 66


In [ ]:
'Ê»' in uniques

False

In [ ]:
len(uniques)

440

In [7]:
with open("cleaned_text.txt", "r", encoding="utf-8") as f:
  lines = f.read().splitlines()

In [ ]:
len(lines)

3011581

In [ ]:
526921719/3011581

174.96514920236248

In [ ]:
lines[:5]

['Sammitda Oʻzbekiston prezidenti Shavkat Mirziyoyev kirish nutqini soʻzladi',
 'Sammitimiz shubhasiz tarixiy voqeadir Bu pandemiya sinovlaridan soʻng uzoq kutilgan doʻstlarning yuzma-yuz uchrashuvidir Bu shaxsiy muloqot ShHT mamlakatlarning kelajagini belgilab beruvchi qoʻshma qarorlar qabul qilish imkoniyatidir - dedi davlat rahbari',
 'Bu tadbir Oʻzbekiston raisligida tarixiy Samarqandda oʻtayotgani biz uchun katta faxrdir Umid qilamanki kecha Samarqand ruhi va xalqimizning koʻp yillik tarixiy madaniyati bilan tanishdingiz - deya Shavkat Mirziyoyevning soʻzlarini keltirmoqda Kunuz muxbiri',
 'Mirziyoyevga koʻra bugungi tadbir doirasida ShHT davlatlari yetakchilari 40 dan ortiq xalqaro kelishuvlarni imzolashi koʻzda tutilgan',
 'Tadbirning qolgan qismi tor doirada davom etmoqda']

In [ ]:
len(lines)

3011581

In [ ]:
# k = np.random.randint(1, 3011500, 50)
# k
# i = 20
# for k_i in k:
#     print(lines[k_i:k_i+i])

In [ ]:
lens = [len(exp) for exp in lines]

In [ ]:
print(max(lens))
print(min(lens))
print(sum(lens)/len(lens))

33701
0
173.965149534414


In [ ]:
tokenizer = Tokenizer(models.Unigram())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.UnigramTrainer(
    vocab_size=20000,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    special_tokens=["<PAD>", "<BOS>", "<EOS>"],
)

In [ ]:
tokenizer.train_from_iterator(lines, trainer=trainer)

In [ ]:
tokenizer.get_vocab_size()

20000

In [ ]:
vocab = tokenizer.get_vocab()
first5 = list(vocab.items())[:5]
print(first5)

[('ĠdoÊ»stlarim', 15479), ('jara', 14382), ('likning', 3217), ('zul', 13811), ('ĠFarhod', 4249)]


In [ ]:
exp = 'ĠdoÊ»stlarim'
exp_id = [15479]
exp2=' doʻstlarim'

In [ ]:
enc2 = tokenizer.encode(exp2)
enc2.ids

[15479]

In [ ]:
tokenizer.decode(enc2.ids)

' doʻstlarim'

In [ ]:
tokenizer.decode(exp_id)

' doʻstlarim'

In [ ]:
enc = tokenizer.encode(exp)
enc.ids

[3, 19862, 19860, 860, 19896, 19979, 19842, 19890, 889, 4427]

In [ ]:
dataset = Dataset.from_dict({"text": lines})
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 3011581
})


In [ ]:
repo_id_for_dataset = "azizdevlab/uzbek_corpus"

# Публикуем датасет
dataset.push_to_hub(repo_id_for_dataset, private=False)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1506 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   4%|3         | 7.86MB /  220MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   2%|1         | 1.57MB / 92.5MB            

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/azizdevlab/uzbek_corpus/commit/85fcdfc31fee032ad7ef8ce5026081956108c26d', commit_message='Upload dataset', commit_description='', oid='85fcdfc31fee032ad7ef8ce5026081956108c26d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/azizdevlab/uzbek_corpus', endpoint='https://huggingface.co', repo_type='dataset', repo_id='azizdevlab/uzbek_corpus'), pr_revision=None, pr_num=None)

In [ ]:
tokenizer.save("my_tokenizer.json")

In [ ]:
api = HfApi()
repo_id = "azizdevlab/gpt2-small-uzbek"
api.upload_file(
    path_or_fileobj="my_tokenizer.json",
    path_in_repo="tokenizer.json",
    repo_id=repo_id,
    repo_type="model"  # тип модели
)

CommitInfo(commit_url='https://huggingface.co/azizdevlab/gpt2-small-uzbek/commit/a6709148bb9751a30f55669f482a3f84eb8df2f3', commit_message='Upload tokenizer.json with huggingface_hub', commit_description='', oid='a6709148bb9751a30f55669f482a3f84eb8df2f3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/azizdevlab/gpt2-small-uzbek', endpoint='https://huggingface.co', repo_type='model', repo_id='azizdevlab/gpt2-small-uzbek'), pr_revision=None, pr_num=None)

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("azizdevlab/gpt2-small-uzbek")

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
import numpy as np

In [15]:
el_ids = np.random.choice(3011581, size=10, replace=False)
print(el_ids)

[ 264641 2445358 1360846  845680 2417466  170995  692872 2045661  445263
  707956]


In [31]:
maxs = []
mins = []

In [78]:
len_tokens = []
el_ids = np.random.choice(3011581, size=100, replace=False)
for id in el_ids:
  len_tokens.append(len(tokenizer(lines[id])['input_ids']))
print(len_tokens)
maxs.append(max(len_tokens))
mins.append(min(len_tokens))

[8, 594, 167, 23, 153, 10, 311, 310, 170, 109, 236, 144, 277, 217, 208, 271, 228, 101, 17, 150, 223, 13, 10, 287, 208, 142, 123, 90, 265, 19, 15, 17, 205, 184, 343, 169, 184, 15, 306, 123, 1329, 100, 148, 180, 11, 283, 8, 14, 11, 260, 163, 19, 16, 257, 241, 239, 464, 184, 10, 530, 144, 139, 151, 228, 348, 269, 19, 16, 15, 156, 117, 129, 314, 12, 22, 17, 116, 33, 248, 18, 126, 23, 24, 7, 375, 377, 116, 99, 15, 17, 174, 8, 163, 464, 235, 124, 831, 376, 305, 15]


In [79]:
print(maxs)

[910, 944, 677, 458, 364, 539, 895, 665, 883, 554, 531, 1756, 975, 581, 1389, 661, 755, 834, 1508, 1329]


In [80]:
mins

[12, 12, 8, 10, 6, 11, 6, 7, 7, 5, 6, 6, 7, 8, 6, 6, 5, 8, 7, 7]